# QSS with `python-control` and AutoReduce

This canonical example uses a simple singularly perturbed system to show how a quasi-steady-state approximation eliminates a fast state while keeping a constant or slowly varying control input. in this example, we demonstrate the compatibility of AutoReduce with the `python-control` package. Particularly, we create a NonlinearIOSystem from `python-control` and use AutoReduce to load this model as a `System` in AutoReduce. We then perform a quasi-steady-state reduction on the system and compare the output of the reduced system to the original system.


## Model

Consider

$$
\dot{x} = -x + z, \qquad \epsilon \dot{z} = x - 2z + u, \qquad 0 < \epsilon \ll 1,
$$

where $u$ is a constant or slowly varying control input. This is the standard singularly perturbed form

$$
\dot{x}=f(x,z,u), \qquad \epsilon\dot{z}=g(x,z,u).
$$


### Create the model using `python-control`

In [1]:
import control as ct
from sympy import simplify, symbols

from autoreduce import solve_timescale_separation
from autoreduce.system.control import from_nonlinear_io_system

x, z, u, epsilon = symbols("x z u epsilon")


def model_update(_t, state, inputs, params):
    x_state, z_state = state
    control_input = inputs[0]
    epsilon_value = params["epsilon"]
    return [
        -x_state + z_state,
        (x_state - 2 * z_state + control_input) / epsilon_value,
    ]


control_system = ct.NonlinearIOSystem(
    model_update,
    None,
    states=["x", "z"],
    inputs=["u"],
    params={"epsilon": 0.01},
)

## Load a NonlinearIOSystem into AutoReduce

In [2]:
system = from_nonlinear_io_system(
    control_system,
    state_symbols=[x, z],
    params=[epsilon],
    params_values=[0.01],
    input_symbols=[u],
    input_values=[1.0],
    x_init=[0.0, 0.0],
)

### Explore the system

In [3]:
system.f

[-x + z, (u + x - 2*z)/epsilon]

## Quasi-steady-state approximation

Because $z$ evolves on the fast time scale, set $\epsilon=0$ in the fast equation. This changes the differential equation into the algebraic constraint

$$
0 = x - 2z + u.
$$

Therefore, the quasi-steady value of $z$ is

$$
z = h(x,u) = \frac{x+u}{2}.
$$


In [4]:
reduced_system, collapsed_system = solve_timescale_separation(
    system,
    slow_states=[x],
    fast_states=[z],
)

[simplify(expr) for expr in reduced_system.f]


Successful solution obtained with states: [x]!


[u/2 - x/2]

## Reduced model

Substituting the quasi-steady value of $z$ into the slow equation gives

$$
\dot{x} = -x + \frac{x+u}{2} = -\frac{1}{2}x + \frac{1}{2}u.
$$

The reduced AutoReduce system stores this one-state model in `reduced_system.x` and `reduced_system.f`.


In [5]:
reduced_system.x, reduced_system.f, collapsed_system.x


([x], [u/2 - x/2], [z])